# Molecular Modeling

**Tier 3 — Module 09** | [Next: Docking →](./02_docking.ipynb)


---[< Previous: From Sequence to Discovery: An Integrative Bioi...](../08_Capstone_Project/01_capstone_project.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Molecular Modeling + Docking: Overview >](01_molecular_modeling_and_docking.ipynb)---

# Molecular Modeling and Docking

**Tier 3 -- Applied Bioinformatics**

Molecular modeling bridges the gap between static structural data and dynamic biological reality. From predicting how a drug binds its target to understanding protein conformational changes, computational molecular modeling is indispensable in modern bioinformatics and drug discovery.

This notebook covers the fundamental theory and practical Python tools for molecular modeling, homology modeling, molecular docking, molecular dynamics, and chemoinformatics. The material is inspired by the molecular modeling curriculum of the Kodomo bioinformatics program (Semester 8, taught by A.V. Golovin, Moscow State University).

**Prerequisites:** Tier 2 (Python, NumPy, pandas, basic structural biology)  
**Libraries:** `rdkit`, `numpy`, `matplotlib`, `pandas`, `mdanalysis` (concepts)  
**External tools discussed:** GROMACS, AutoDock Vina, SWISS-MODEL, Modeller, PyMOL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 1. Introduction to Molecular Modeling

### 1.1 Why Model Biomolecules?

Experimental structure determination (X-ray crystallography, cryo-EM, NMR) provides snapshots, but biomolecules are dynamic. Molecular modeling lets us:

| Application | What modeling provides | Example |
|------------|----------------------|--------|
| **Drug discovery** | Predict binding poses and affinities | Virtual screening of millions of compounds |
| **Protein engineering** | Assess effects of mutations | Designing thermostable enzymes |
| **Mechanism understanding** | Visualize conformational changes | Ion channel gating, enzyme catalysis |
| **Structure prediction** | Build models when no crystal structure exists | Homology models for uncharacterized proteins |
| **Materials design** | Predict properties of biomaterials | Self-assembling peptide nanostructures |

### 1.2 Levels of Theory

Molecular modeling spans multiple levels of approximation. The choice depends on the system size and the question being asked.

```
Accuracy ▲                                    System size ▲
         │  Quantum Mechanics (QM)                       │  Coarse-Grained (CG)
         │  ├─ Ab initio (HF, DFT, MP2)                 │  ├─ MARTINI
         │  ├─ Semi-empirical (AM1, PM3)                 │  ├─ 4:1 mapping (4 atoms → 1 bead)
         │  └─ ~100 atoms max                            │  └─ Millions of particles, μs timescales
         │                                               │
         │  Molecular Mechanics (MM)                     │  Continuum / Implicit solvent
         │  ├─ Classical force fields                    │  ├─ Poisson-Boltzmann
         │  ├─ AMBER, CHARMM, OPLS                      │  ├─ Generalized Born
         │  └─ ~10⁵–10⁶ atoms, ns timescales            │  └─ Entire organelles
         ▼                                               ▼
```

**Key insight:** There is always a tradeoff between accuracy and computational cost. Quantum mechanics treats electrons explicitly but is limited to small systems. Molecular mechanics uses parametrized potentials (force fields) and can handle entire proteins in explicit solvent. Coarse-grained models sacrifice atomic detail for access to longer timescales.

### 1.3 Quantum Mechanics in Brief

Quantum mechanical methods solve (approximately) the Schrödinger equation:

$$\hat{H}\Psi = E\Psi$$

- **Ab initio methods** (Hartree-Fock, post-HF): No empirical parameters; computationally expensive.
- **Density Functional Theory (DFT):** Reformulates the problem in terms of electron density rather than the wavefunction. More efficient, widely used for systems of ~100–500 atoms.
- **Semi-empirical methods** (AM1, PM3, PM7): Use empirical parameters to speed up calculations. Useful for quick estimates.

QM is essential for:
- Chemical reactions (bond breaking/forming)
- Accurate charge distributions
- Parameterizing force fields
- QM/MM hybrid methods (quantum region embedded in a classical environment)

In [ ]:
# Visualization: the Lennard-Jones potential (a core non-bonded interaction)
# This is one of the fundamental functions in molecular mechanics

def lennard_jones(r, epsilon=1.0, sigma=1.0):
    """Compute Lennard-Jones potential: V(r) = 4*epsilon*((sigma/r)^12 - (sigma/r)^6)"""
    return 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)

r = np.linspace(0.9, 3.0, 500)
V = lennard_jones(r)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r, V, 'b-', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Distance r (in units of σ)')
ax.set_ylabel('Potential Energy V(r) (in units of ε)')
ax.set_title('Lennard-Jones Potential')
ax.set_ylim(-1.5, 3)
ax.set_xlim(0.85, 3.0)

# Annotate equilibrium distance and well depth
r_min = 2**(1/6)  # equilibrium distance
ax.annotate(f'Equilibrium: r = 2^(1/6)σ ≈ {r_min:.3f}σ',
            xy=(r_min, -1), xytext=(1.8, -0.5),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red')
ax.annotate('Repulsive\n(r⁻¹² dominates)', xy=(1.0, 2.0), fontsize=10, color='darkred')
ax.annotate('Attractive\n(r⁻⁶ dominates)', xy=(2.0, -0.3), fontsize=10, color='darkblue')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Force Fields and Molecular Mechanics

### 2.1 The Force Field Concept

A **force field** is a mathematical function and associated parameter set that describes the potential energy of a molecular system. In molecular mechanics, the total energy is decomposed into bonded and non-bonded terms:

$$E_{\text{total}} = E_{\text{bonds}} + E_{\text{angles}} + E_{\text{dihedrals}} + E_{\text{electrostatic}} + E_{\text{van der Waals}}$$

### 2.2 Bonded Interactions

**Bond stretching** (harmonic approximation):
$$E_{\text{bond}} = \frac{1}{2} k_b (r - r_0)^2$$

**Angle bending:**
$$E_{\text{angle}} = \frac{1}{2} k_\theta (\theta - \theta_0)^2$$

**Dihedral (torsional) angles:**
$$E_{\text{dihedral}} = \sum_n V_n [1 + \cos(n\phi - \gamma)]$$

Where:
- $k_b$, $k_\theta$ are force constants
- $r_0$, $\theta_0$ are equilibrium values
- $V_n$ is the torsional barrier height
- $n$ is the periodicity, $\gamma$ is the phase

In [ ]:
# Visualize the four main force field terms

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Bond stretching
r = np.linspace(0.8, 2.0, 300)
r0, kb = 1.5, 300  # equilibrium distance (Å), force constant (kcal/mol/Å²)
E_bond = 0.5 * kb * (r - r0)**2
axes[0, 0].plot(r, E_bond, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Bond length r (Å)')
axes[0, 0].set_ylabel('Energy (kcal/mol)')
axes[0, 0].set_title('Bond Stretching (Harmonic)')
axes[0, 0].axvline(x=r0, color='r', linestyle='--', label=f'r₀ = {r0} Å')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Angle bending
theta = np.linspace(90, 150, 300)
theta0, ka = 120, 50
E_angle = 0.5 * ka * (np.radians(theta) - np.radians(theta0))**2
axes[0, 1].plot(theta, E_angle, 'g-', linewidth=2)
axes[0, 1].set_xlabel('Bond angle θ (degrees)')
axes[0, 1].set_ylabel('Energy (kcal/mol)')
axes[0, 1].set_title('Angle Bending (Harmonic)')
axes[0, 1].axvline(x=theta0, color='r', linestyle='--', label=f'θ₀ = {theta0}°')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Dihedral potential
phi = np.linspace(-180, 180, 300)
V1, V2, V3 = 0.5, 0.2, 0.8
E_dih = V1*(1 + np.cos(np.radians(phi))) + V2*(1 - np.cos(2*np.radians(phi))) + V3*(1 + np.cos(3*np.radians(phi)))
axes[1, 0].plot(phi, E_dih, 'm-', linewidth=2)
axes[1, 0].set_xlabel('Dihedral angle φ (degrees)')
axes[1, 0].set_ylabel('Energy (kcal/mol)')
axes[1, 0].set_title('Dihedral (Torsional) Potential')
axes[1, 0].grid(True, alpha=0.3)

# Non-bonded: Lennard-Jones + Coulomb
r_nb = np.linspace(2.5, 10.0, 300)
sigma, epsilon = 3.4, 0.24  # Å, kcal/mol (typical for carbon)
E_lj = 4 * epsilon * ((sigma/r_nb)**12 - (sigma/r_nb)**6)
# Coulomb with opposite charges
q1, q2 = 0.5, -0.5  # partial charges (e)
k_coul = 332.0637  # kcal·Å/(mol·e²)
E_coul = k_coul * q1 * q2 / r_nb
axes[1, 1].plot(r_nb, E_lj, 'b-', linewidth=2, label='van der Waals (LJ)')
axes[1, 1].plot(r_nb, E_coul, 'r-', linewidth=2, label='Electrostatic (Coulomb)')
axes[1, 1].plot(r_nb, E_lj + E_coul, 'k--', linewidth=2, label='Total non-bonded')
axes[1, 1].set_xlabel('Distance r (Å)')
axes[1, 1].set_ylabel('Energy (kcal/mol)')
axes[1, 1].set_title('Non-bonded Interactions')
axes[1, 1].set_ylim(-2, 2)
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Common Force Fields

| Force Field | Developers | Typical Use | Key Features |
|------------|-----------|-------------|---------------|
| **AMBER** | Kollman group (UCSF) | Proteins, nucleic acids | Widely used for biomolecular MD; includes ff14SB, ff19SB |
| **CHARMM** | Karplus group (Harvard) | Proteins, lipids, small molecules | CGenFF for drug-like molecules; extensive lipid parameters |
| **OPLS-AA** | Jorgensen group (Yale) | Proteins, organic molecules | Optimized for liquid-state thermodynamics |
| **GROMOS** | van Gunsteren group (ETH) | Proteins, lipids | United-atom model (no aliphatic H); efficient |
| **MARTINI** | Marrink group (Groningen) | Coarse-grained simulations | 4:1 heavy atom mapping; membranes, large systems |

**Important:** A force field is only as good as its parametrization. Never mix parameters from different force fields. Always validate your system setup.

### 2.4 Energy Minimization

Before any simulation, we need to remove bad contacts (steric clashes) by minimizing the potential energy. Common algorithms:

- **Steepest Descent:** Moves along the negative gradient. Robust but slow near the minimum. Good for initial relaxation.
- **Conjugate Gradient:** Uses information from previous steps to avoid oscillation. Faster convergence near the minimum.
- **L-BFGS:** Quasi-Newton method that approximates the Hessian. Fastest for smooth energy surfaces.

In GROMACS, energy minimization is typically performed with steepest descent until the maximum force drops below a threshold (e.g., 1000 kJ/mol/nm), then optionally refined with conjugate gradient.

In [ ]:
# Demonstrate steepest descent vs conjugate gradient on a 2D surface

def rosenbrock(x, y, a=1, b=100):
    """Rosenbrock function -- a classic optimization test surface."""
    return (a - x)**2 + b * (y - x**2)**2

def rosenbrock_grad(x, y, a=1, b=100):
    dx = -2*(a - x) + b * 2 * (y - x**2) * (-2*x)
    dy = b * 2 * (y - x**2)
    return np.array([dx, dy])

def steepest_descent(x0, y0, lr=0.001, steps=500):
    path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(steps):
        grad = rosenbrock_grad(x, y)
        x -= lr * grad[0]
        y -= lr * grad[1]
        path.append((x, y))
    return np.array(path)

# Run steepest descent
path_sd = steepest_descent(-1.5, 2.0, lr=0.001, steps=3000)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
x_grid = np.linspace(-2, 2, 200)
y_grid = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = rosenbrock(X, Y)

cs = ax.contour(X, Y, Z, levels=np.logspace(-0.5, 3.5, 20), cmap='viridis')
ax.plot(path_sd[:, 0], path_sd[:, 1], 'r.-', markersize=1, linewidth=0.5, alpha=0.7, label='Steepest descent path')
ax.plot(-1.5, 2.0, 'ro', markersize=10, label='Start')
ax.plot(1, 1, 'g*', markersize=15, label='Minimum (1, 1)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Energy Minimization: Steepest Descent on Rosenbrock Surface')
ax.legend()
plt.colorbar(cs, label='Energy')
plt.tight_layout()
plt.show()

print(f"Final position: ({path_sd[-1, 0]:.4f}, {path_sd[-1, 1]:.4f})")
print(f"Final energy:    {rosenbrock(path_sd[-1, 0], path_sd[-1, 1]):.6f}")
print(f"True minimum:    (1.0, 1.0) with energy 0.0")

---
## 3. Homology Modeling

### 3.1 The Principle

Proteins with similar sequences tend to adopt similar 3D structures. **Homology modeling** (or comparative modeling) exploits this to predict the structure of a protein (target) using a known structure (template) as a scaffold.

The general workflow:

```
Target sequence
       │
       ▼
┌─────────────────┐
│ Template search  │  ← BLAST, HHpred, SWISS-MODEL
│ (find homologs)  │
└────────┬────────┘
         ▼
┌─────────────────┐
│ Target-template  │  ← Sequence alignment (critical step!)
│ alignment        │
└────────┬────────┘
         ▼
┌─────────────────┐
│ Model building   │  ← Copy coordinates, build loops, add side chains
│                  │
└────────┬────────┘
         ▼
┌─────────────────┐
│ Refinement       │  ← Energy minimization, MD relaxation
└────────┬────────┘
         ▼
┌─────────────────┐
│ Quality check    │  ← Ramachandran, QMEAN, ProSA
└─────────────────┘
```

### 3.2 Template Selection

The quality of a homology model depends primarily on:

1. **Sequence identity** between target and template:
   - \>50%: High-quality model (comparable to medium-resolution crystal structure)
   - 30--50%: Reasonable backbone, some loop/side-chain errors
   - <30%: "Twilight zone" -- alignment errors dominate; use fold recognition (threading) methods

2. **Template quality:** Resolution, R-factor, completeness
3. **Functional relevance:** Same oligomeric state, similar ligands bound
4. **Coverage:** Template should cover the full target, especially functional regions

**Tools for template search:**
- **BLAST/PSI-BLAST:** Sequence-based search of PDB
- **HHpred:** Profile-profile comparison (more sensitive in the twilight zone)
- **SWISS-MODEL Template Library:** Curated, pre-computed alignments

### 3.3 Model Quality Assessment

Never trust a model blindly. Always validate:

| Metric | What it checks | Tool |
|--------|---------------|------|
| **Ramachandran plot** | Backbone φ/ψ angles in allowed regions | PROCHECK, MolProbity |
| **QMEAN** | Composite score: Cβ interaction, all-atom, solvation, torsion | SWISS-MODEL |
| **ProSA Z-score** | Overall model quality vs known structures | ProSA-web |
| **DOPE score** | Statistical potential per residue | Modeller |
| **Verify3D** | Compatibility of each residue with its environment | UCLA-DOE server |

In [ ]:
# Simulate a Ramachandran plot to illustrate model validation
# In practice, these values come from PDB file parsing

# Generate synthetic backbone angles for a "good" protein model
n_residues = 200

# Alpha-helix region (φ ≈ -60, ψ ≈ -45)
n_helix = 80
phi_helix = np.random.normal(-60, 8, n_helix)
psi_helix = np.random.normal(-45, 8, n_helix)

# Beta-sheet region (φ ≈ -120, ψ ≈ +130)
n_sheet = 60
phi_sheet = np.random.normal(-120, 10, n_sheet)
psi_sheet = np.random.normal(130, 10, n_sheet)

# Left-handed helix region (φ ≈ +60, ψ ≈ +45) -- rare
n_left = 5
phi_left = np.random.normal(60, 10, n_left)
psi_left = np.random.normal(45, 10, n_left)

# Loop regions (more dispersed)
n_loop = n_residues - n_helix - n_sheet - n_left
phi_loop = np.random.normal(-80, 30, n_loop)
psi_loop = np.random.normal(0, 50, n_loop)

# Outliers ("bad" residues)
n_outlier = 3
phi_outlier = np.random.uniform(0, 120, n_outlier)
psi_outlier = np.random.uniform(-50, 50, n_outlier)

phi_all = np.concatenate([phi_helix, phi_sheet, phi_left, phi_loop, phi_outlier])
psi_all = np.concatenate([psi_helix, psi_sheet, psi_left, psi_loop, psi_outlier])

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(phi_all[:-n_outlier], psi_all[:-n_outlier], s=8, c='steelblue', alpha=0.6, label='Allowed')
ax.scatter(phi_outlier, psi_outlier, s=30, c='red', marker='x', linewidths=2, label='Outliers')

# Draw approximate allowed regions
from matplotlib.patches import Ellipse
ell_helix = Ellipse((-60, -45), 50, 50, fill=False, edgecolor='green', linewidth=2, linestyle='--', label='α-helix region')
ell_sheet = Ellipse((-120, 130), 60, 60, fill=False, edgecolor='orange', linewidth=2, linestyle='--', label='β-sheet region')
ax.add_patch(ell_helix)
ax.add_patch(ell_sheet)

ax.set_xlabel('φ (phi) degrees')
ax.set_ylabel('ψ (psi) degrees')
ax.set_title('Ramachandran Plot (Simulated)')
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.2)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.2)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.2)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

outlier_pct = n_outlier / (len(phi_all)) * 100
print(f"Residues in outlier regions: {n_outlier}/{len(phi_all)} ({outlier_pct:.1f}%)")
print("A good model should have >90% of residues in favored regions and <0.5% outliers.")

### 3.4 AlphaFold and the Prediction Revolution

Since 2020, **AlphaFold2** (DeepMind) has transformed structure prediction. Key points:

- Uses deep learning on multiple sequence alignments and structural templates
- Achieves near-experimental accuracy for many proteins (median GDT-TS > 90 at CASP14)
- AlphaFold Protein Structure Database provides >200 million predicted structures
- **Limitations:** Struggles with conformational ensembles, intrinsically disordered regions, protein-protein complexes (partially addressed by AlphaFold-Multimer), and novel folds with no homologs in training data
- Always check the **pLDDT score** (per-residue confidence): >90 = high confidence, 70--90 = moderate, <50 = low (likely disordered)

---
## 4. Molecular Docking

### 4.1 What is Molecular Docking?

Molecular docking predicts how a small molecule (ligand) binds to a macromolecule (receptor), estimating:
1. **Binding pose** (orientation and conformation of the ligand in the binding site)
2. **Binding affinity** (how strongly the ligand binds, approximated by a scoring function)

Applications:
- **Virtual screening:** Test millions of compounds computationally before synthesis
- **Lead optimization:** Understand structure-activity relationships
- **Target identification:** Where does a known drug bind?
- **Protein-protein docking:** Predict complex structures

### 4.2 Rigid vs. Flexible Docking

| Approach | Receptor | Ligand | Speed | Accuracy |
|---------|---------|--------|-------|----------|
| **Rigid-rigid** | Fixed | Fixed | Fastest | Lowest -- misses induced fit |
| **Rigid receptor, flexible ligand** | Fixed | Torsions sampled | Fast | Good for many cases |
| **Flexible receptor** | Selected side chains move | Flexible | Slow | Best for induced fit |
| **Ensemble docking** | Multiple receptor conformations | Flexible | Slow | Captures receptor flexibility indirectly |

Most docking programs (AutoDock Vina, Glide, GOLD) use **rigid receptor with flexible ligand** as the default.

### 4.3 Scoring Functions

Scoring functions estimate binding affinity. Three main classes:

1. **Force field-based:** Use molecular mechanics energy terms (van der Waals, electrostatics) + desolvation penalty. Example: AutoDock.

2. **Empirical:** Weighted sum of terms (H-bonds, hydrophobic contacts, entropy penalty) fitted to experimental binding data. Example: AutoDock Vina, ChemPLP (GOLD).

3. **Knowledge-based:** Derived from statistical analysis of atom-pair frequencies in known protein-ligand complexes. Example: DrugScore, PMF.

**Key limitation:** Scoring functions are approximate. They often correctly rank poses for the same ligand but struggle to rank different ligands against each other. Always validate top predictions experimentally.

### 4.4 AutoDock Vina Workflow

AutoDock Vina is one of the most widely used docking programs. A typical workflow:

```bash
# 1. Prepare receptor (remove water, add hydrogens, assign charges)
#    Output: receptor.pdbqt
prepare_receptor4.py -r protein.pdb -o receptor.pdbqt

# 2. Prepare ligand (generate 3D coordinates, add hydrogens)
#    Output: ligand.pdbqt
prepare_ligand4.py -l ligand.mol2 -o ligand.pdbqt

# 3. Define search box (grid centered on binding site)
#    config.txt:
#    center_x = 15.0
#    center_y = 20.0
#    center_z = 25.0
#    size_x = 20
#    size_y = 20
#    size_z = 20

# 4. Run docking
vina --receptor receptor.pdbqt --ligand ligand.pdbqt --config config.txt --out output.pdbqt

# 5. Analyze results (poses ranked by score in kcal/mol)
#    More negative = stronger predicted binding
```

### 4.5 Binding Site Prediction

If no co-crystal structure is available, you need to predict where ligands bind:

- **Geometry-based:** Find cavities/pockets on the protein surface (fpocket, SiteMap, CASTp)
- **Energy-based:** Probe the surface with chemical groups, find favorable interaction sites (FTMap)
- **Homology-based:** Transfer binding site from a homologous protein
- **Conservation-based:** Functionally important residues are often conserved and cluster at binding sites

In [ ]:
# Simulate and parse AutoDock Vina output
# In practice, this would come from actual docking runs

vina_output = """mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.7      0.000      0.000
   2       -8.3      1.245      2.187
   3       -7.9      2.056      4.321
   4       -7.6      1.876      3.987
   5       -7.2      5.432      8.654
   6       -6.8      3.210      6.543
   7       -6.5      6.789      9.876
   8       -6.1      7.234     10.432
   9       -5.8      8.432     12.345
"""

def parse_vina_output(text):
    """Parse AutoDock Vina log output into a DataFrame."""
    rows = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line and line[0].isdigit():
            parts = line.split()
            rows.append({
                'mode': int(parts[0]),
                'affinity_kcal_mol': float(parts[1]),
                'rmsd_lb': float(parts[2]),
                'rmsd_ub': float(parts[3])
            })
    return pd.DataFrame(rows)

df_vina = parse_vina_output(vina_output)
print("Parsed Vina docking results:")
print(df_vina.to_string(index=False))

print(f"\nBest binding affinity: {df_vina['affinity_kcal_mol'].min()} kcal/mol")
print(f"Number of distinct binding modes (RMSD > 4 Å from best): "
      f"{(df_vina['rmsd_ub'] > 4.0).sum() + 1}")

In [ ]:
# Visualize docking scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score vs mode
axes[0].bar(df_vina['mode'], df_vina['affinity_kcal_mol'], color='steelblue', edgecolor='navy')
axes[0].set_xlabel('Docking Mode')
axes[0].set_ylabel('Binding Affinity (kcal/mol)')
axes[0].set_title('AutoDock Vina: Docking Scores by Mode')
axes[0].axhline(y=-7.0, color='red', linestyle='--', alpha=0.7, label='Threshold (-7.0 kcal/mol)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# RMSD vs affinity
sc = axes[1].scatter(df_vina['rmsd_ub'], df_vina['affinity_kcal_mol'],
                     c=df_vina['mode'], cmap='viridis', s=100, edgecolors='black')
axes[1].set_xlabel('RMSD from Best Mode (Å)')
axes[1].set_ylabel('Binding Affinity (kcal/mol)')
axes[1].set_title('Pose Diversity vs. Score')
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[1], label='Mode number')

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Modes 1-4 cluster near the same binding site (low RMSD).")
print("- Mode 5+ explore alternative binding sites (high RMSD).")
print("- The best 3 modes all score < -7.5 kcal/mol, suggesting a reliable binding pose.")

---
## 5. Molecular Dynamics Fundamentals

### 5.1 What is Molecular Dynamics?

**Molecular Dynamics (MD)** simulates the time evolution of a molecular system by numerically integrating Newton's equations of motion:

$$F_i = m_i \cdot a_i = -\nabla_i V$$

where $F_i$ is the force on atom $i$, $m_i$ its mass, $a_i$ its acceleration, and $V$ the potential energy function (force field).

MD provides a trajectory: a time series of atomic positions and velocities from which we can compute thermodynamic and kinetic properties.

### 5.2 Integration Algorithms

**Verlet algorithm:**
$$\mathbf{r}(t + \Delta t) = 2\mathbf{r}(t) - \mathbf{r}(t - \Delta t) + \frac{\mathbf{F}(t)}{m}\Delta t^2$$

- Time-reversible, symplectic (conserves energy well)
- Velocities not explicitly computed (but can be derived)

**Leap-frog algorithm** (used in GROMACS):
$$\mathbf{v}(t + \frac{\Delta t}{2}) = \mathbf{v}(t - \frac{\Delta t}{2}) + \frac{\mathbf{F}(t)}{m}\Delta t$$
$$\mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \mathbf{v}(t + \frac{\Delta t}{2})\Delta t$$

- Positions and velocities computed at staggered half-steps
- Equivalent to Verlet in terms of trajectory
- Easier to implement thermostats

**Typical timestep:** 1--2 fs (femtoseconds). With constraints on bonds involving hydrogen (LINCS/SHAKE), 2 fs is standard.

In [ ]:
# Demonstrate the Verlet integrator on a simple harmonic oscillator

def verlet_harmonic(k=1.0, m=1.0, x0=1.0, v0=0.0, dt=0.05, n_steps=500):
    """Simulate a 1D harmonic oscillator using Verlet integration."""
    x = np.zeros(n_steps)
    v = np.zeros(n_steps)
    x[0] = x0
    v[0] = v0
    
    # First step using velocity Verlet
    a = -k * x[0] / m
    for i in range(1, n_steps):
        x[i] = x[i-1] + v[i-1]*dt + 0.5*a*dt**2
        a_new = -k * x[i] / m
        v[i] = v[i-1] + 0.5*(a + a_new)*dt
        a = a_new
    
    t = np.arange(n_steps) * dt
    return t, x, v

t, x_verlet, v_verlet = verlet_harmonic(k=1.0, m=1.0, dt=0.1, n_steps=500)

# Exact solution
omega = 1.0  # sqrt(k/m)
x_exact = np.cos(omega * t)

# Energy conservation check
KE = 0.5 * v_verlet**2
PE = 0.5 * x_verlet**2
TE = KE + PE

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(t, x_verlet, 'b-', linewidth=1.5, label='Verlet')
axes[0].plot(t, x_exact, 'r--', linewidth=1.5, label='Exact')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Position')
axes[0].set_title('Harmonic Oscillator: Position')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_verlet, v_verlet, 'g-', linewidth=0.5)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Velocity')
axes[1].set_title('Phase Space (should be a circle)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, KE, label='Kinetic', alpha=0.7)
axes[2].plot(t, PE, label='Potential', alpha=0.7)
axes[2].plot(t, TE, 'k-', linewidth=2, label='Total')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Energy')
axes[2].set_title('Energy Conservation')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total energy drift: {abs(TE[-1] - TE[0]):.6e}")
print("The Verlet integrator conserves energy to machine precision over long trajectories.")

### 5.3 Periodic Boundary Conditions (PBC)

To simulate bulk properties without surface effects, we replicate the simulation box infinitely in all directions:

```
┌─────────┬─────────┬─────────┐
│ image   │ image   │ image   │
│         │         │         │
├─────────┼─────────┼─────────┤
│ image   │ CENTRAL │ image   │
│         │   BOX   │         │
├─────────┼─────────┼─────────┤
│ image   │ image   │ image   │
│         │         │         │
└─────────┴─────────┴─────────┘
```

- When a molecule exits one side of the box, it re-enters from the opposite side
- Interactions are computed with the **minimum image convention**: each atom interacts with the closest periodic image
- The box must be large enough that a molecule does not interact with its own periodic image
- Common box shapes: cubic, truncated octahedron (more efficient for globular proteins), rhombic dodecahedron

### 5.4 Thermostats and Barostats

To simulate at constant temperature (NVT) or constant pressure (NPT), we need thermostats and barostats:

**Thermostats** (temperature control):

| Thermostat | Method | Pros | Cons |
|-----------|--------|------|------|
| **Berendsen** | Rescale velocities toward target T | Fast equilibration | Does not generate canonical ensemble |
| **V-rescale** | Berendsen + stochastic term | Correct ensemble, fast | Default in GROMACS |
| **Nosé-Hoover** | Extended Lagrangian | Rigorous canonical ensemble | Can oscillate; needs chain thermostats |
| **Langevin** | Add friction + random forces | Correct ensemble | Alters dynamics (no true NVE limit) |

**Barostats** (pressure control):

| Barostat | Method | Use case |
|---------|--------|----------|
| **Berendsen** | Rescale box dimensions | Equilibration only |
| **Parrinello-Rahman** | Extended Lagrangian | Production NPT runs |
| **MTTK** | Martyna-Tuckerman-Tobias-Klein | Rigorous; used with Nosé-Hoover |

### 5.5 GROMACS Workflow

GROMACS is one of the fastest and most widely used MD engines for biomolecular simulation. A typical protein-in-water simulation workflow:

```bash
# Step 1: Generate topology
gmx pdb2gmx -f protein.pdb -o protein.gro -water tip3p -ff amber99sb-ildn

# Step 2: Define simulation box
gmx editconf -f protein.gro -o box.gro -c -d 1.0 -bt dodecahedron

# Step 3: Solvate
gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top

# Step 4: Add ions (neutralize charge)
gmx grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr
gmx genion -s ions.tpr -o ions.gro -p topol.top -pname NA -nname CL -neutral

# Step 5: Energy minimization
gmx grompp -f em.mdp -c ions.gro -p topol.top -o em.tpr
gmx mdrun -deffnm em

# Step 6: NVT equilibration (100 ps)
gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr
gmx mdrun -deffnm nvt

# Step 7: NPT equilibration (100 ps)
gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -p topol.top -o npt.tpr
gmx mdrun -deffnm npt

# Step 8: Production MD (e.g., 100 ns)
gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr
gmx mdrun -deffnm md
```

### 5.6 MD Analysis Metrics

After a simulation, we analyze the trajectory to extract biological insights:

| Metric | What it tells you | GROMACS tool |
|--------|-------------------|-------------|
| **RMSD** | Overall structural drift from reference | `gmx rms` |
| **RMSF** | Per-residue flexibility (B-factor equivalent) | `gmx rmsf` |
| **Radius of gyration (Rg)** | Protein compactness | `gmx gyrate` |
| **Hydrogen bonds** | Stability of specific interactions | `gmx hbond` |
| **Secondary structure** | DSSP assignment over time | `gmx do_dssp` |
| **Solvent Accessible Surface Area** | Exposure of residues | `gmx sasa` |
| **Principal Component Analysis** | Dominant motions | `gmx covar`, `gmx anaeig` |
| **Free energy (PMF)** | Energy landscapes | `gmx wham` |

In [ ]:
# Simulate typical MD analysis results for educational purposes

# Generate synthetic RMSD data
time_ns = np.linspace(0, 100, 5000)
# RMSD rises initially then plateaus (typical equilibrated simulation)
rmsd = 0.15 * (1 - np.exp(-time_ns / 5)) + 0.02 * np.random.randn(len(time_ns))
rmsd = np.clip(rmsd, 0, None)

# Radius of gyration (should be stable for a folded protein)
rg = 1.45 + 0.015 * np.random.randn(len(time_ns))

# Per-residue RMSF
n_residues = 150
residue_ids = np.arange(1, n_residues + 1)
# Low RMSF for core, higher for loops (around residues 30-40, 80-95, and termini)
rmsf_base = 0.06 * np.ones(n_residues)
rmsf_base[:5] = 0.25  # N-terminus
rmsf_base[-5:] = 0.30  # C-terminus
rmsf_base[28:42] = 0.18  # Loop 1
rmsf_base[78:96] = 0.22  # Loop 2
rmsf = rmsf_base + 0.02 * np.random.rand(n_residues)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# RMSD
axes[0, 0].plot(time_ns, rmsd, 'b-', linewidth=0.5, alpha=0.7)
# Running average
window = 100
rmsd_smooth = np.convolve(rmsd, np.ones(window)/window, mode='valid')
axes[0, 0].plot(time_ns[window-1:], rmsd_smooth, 'r-', linewidth=2, label='Running average')
axes[0, 0].set_xlabel('Time (ns)')
axes[0, 0].set_ylabel('RMSD (nm)')
axes[0, 0].set_title('Backbone RMSD vs. Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Rg
axes[0, 1].plot(time_ns, rg, 'g-', linewidth=0.5, alpha=0.7)
rg_smooth = np.convolve(rg, np.ones(window)/window, mode='valid')
axes[0, 1].plot(time_ns[window-1:], rg_smooth, 'r-', linewidth=2, label='Running average')
axes[0, 1].set_xlabel('Time (ns)')
axes[0, 1].set_ylabel('Rg (nm)')
axes[0, 1].set_title('Radius of Gyration vs. Time')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# RMSF
axes[1, 0].plot(residue_ids, rmsf, 'purple', linewidth=1.5)
axes[1, 0].fill_between(residue_ids, rmsf, alpha=0.3, color='purple')
axes[1, 0].axhline(y=0.15, color='red', linestyle='--', alpha=0.5, label='Flexibility threshold')
axes[1, 0].set_xlabel('Residue Number')
axes[1, 0].set_ylabel('RMSF (nm)')
axes[1, 0].set_title('Per-Residue RMSF (Backbone Flexibility)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# H-bonds over time
hbonds = np.random.poisson(lam=85, size=len(time_ns))
axes[1, 1].plot(time_ns, hbonds, 'orange', linewidth=0.3, alpha=0.5)
hb_smooth = np.convolve(hbonds, np.ones(window)/window, mode='valid')
axes[1, 1].plot(time_ns[window-1:], hb_smooth, 'red', linewidth=2, label='Running average')
axes[1, 1].set_xlabel('Time (ns)')
axes[1, 1].set_ylabel('Number of H-bonds')
axes[1, 1].set_title('Intramolecular Hydrogen Bonds')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation:")
print(f"- RMSD equilibrates after ~10 ns, plateauing at ~{rmsd_smooth[-1]:.2f} nm")
print(f"- Rg is stable at ~{rg_smooth[-1]:.2f} nm (protein remains compact)")
print(f"- Loops (residues 29-41, 79-95) and termini show highest flexibility")
print(f"- H-bond count is stable (~{hb_smooth[-1]:.0f}), indicating structural integrity")

---
## 6. Chemoinformatics Basics

### 6.1 SMILES Notation

**SMILES** (Simplified Molecular-Input Line-Entry System) represents molecular structures as text strings:

| Molecule | SMILES | Key features |
|---------|--------|-------------|
| Methane | `C` | Single atom |
| Ethanol | `CCO` | Chain, implicit H |
| Acetic acid | `CC(=O)O` | Branching with `()`, double bond `=` |
| Benzene | `c1ccccc1` | Aromatic ring (lowercase), ring closure `1` |
| Aspirin | `CC(=O)Oc1ccccc1C(=O)O` | Mixed aromatic/aliphatic |
| Caffeine | `Cn1cnc2c1c(=O)n(c(=O)n2C)C` | Fused heterocycles |
| ATP | `c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N` | Stereochemistry `@@` |

**SMILES rules:**
- Atoms in brackets `[]` when non-standard valence or charges: `[NH4+]`
- `=` double bond, `#` triple bond
- `/` and `\` for cis/trans geometry
- `@`/`@@` for tetrahedral stereochemistry
- Ring closures: matching digits mark ring bonds

In [ ]:
# RDKit: the standard Python library for chemoinformatics
# Install: pip install rdkit-pypi  or  conda install -c conda-forge rdkit

try:
    from rdkit import Chem
    from rdkit.Chem import Draw, Descriptors, AllChem, DataStructs
    from rdkit.Chem import rdMolDescriptors
    RDKIT_AVAILABLE = True
    print("RDKit is available!")
except ImportError:
    RDKIT_AVAILABLE = False
    print("RDKit not installed. Install with: pip install rdkit-pypi")
    print("Code cells below will show expected outputs as comments.")

In [ ]:
# Working with molecules in RDKit

if RDKIT_AVAILABLE:
    # Create molecules from SMILES
    molecules = {
        'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
        'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
        'Caffeine': 'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
        'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
        'Penicillin G': 'CC1(C(N2C(S1)C(C2=O)NC(=O)Cc3ccccc3)C(=O)O)C',
        'Metformin': 'CN(C)C(=N)NC(=N)N',
    }

    mols = []
    names = []
    for name, smi in molecules.items():
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            mols.append(mol)
            names.append(name)

    # Draw a grid of molecules
    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(350, 250), legends=names)
    display(img)
else:
    print("[Would display a grid of 6 drug molecules: Aspirin, Ibuprofen, Caffeine,")
    print(" Paracetamol, Penicillin G, and Metformin]")

### 6.2 Molecular Descriptors

Molecular descriptors are numerical values that characterize a molecule's physical, chemical, or structural properties. They are essential for:
- QSAR/QSPR modeling (predicting activity/properties from structure)
- Virtual screening
- Clustering and diversity analysis

Common descriptors:
- **Molecular weight (MW):** Total atomic mass
- **LogP:** Octanol-water partition coefficient (lipophilicity)
- **TPSA:** Topological polar surface area (relates to membrane permeability)
- **H-bond donors/acceptors:** Key for intermolecular interactions
- **Rotatable bonds:** Molecular flexibility
- **Aromatic rings:** Planarity, pi-stacking potential

In [ ]:
# Calculate molecular descriptors for our drug set

if RDKIT_AVAILABLE:
    descriptor_data = []
    for name, mol in zip(names, mols):
        descriptor_data.append({
            'Drug': name,
            'MW': round(Descriptors.MolWt(mol), 1),
            'LogP': round(Descriptors.MolLogP(mol), 2),
            'TPSA': round(Descriptors.TPSA(mol), 1),
            'HBD': Descriptors.NumHDonors(mol),
            'HBA': Descriptors.NumHAcceptors(mol),
            'RotBonds': Descriptors.NumRotatableBonds(mol),
            'Rings': Descriptors.RingCount(mol),
        })

    df_desc = pd.DataFrame(descriptor_data)
    print("Molecular Descriptors:")
    print(df_desc.to_string(index=False))
else:
    # Show expected output
    print("Molecular Descriptors (expected output):")
    print("       Drug     MW  LogP  TPSA  HBD  HBA  RotBonds  Rings")
    print("    Aspirin  180.2  1.31  63.6    1    4         2      1")
    print("  Ibuprofen  206.3  3.50  37.3    1    2         4      1")
    print("   Caffeine  194.2 -0.07  58.4    0    6         0      2")
    print("Paracetamol  151.2  0.91  49.3    2    3         1      1")
    print("Penicillin G 334.4  1.83  87.9    2    5         4      3")
    print("  Metformin  129.2 -1.36  91.5    3    5         2      0")

### 6.3 Molecular Fingerprints and Similarity

**Molecular fingerprints** encode structural features as bit vectors. They enable fast similarity searching across large chemical databases.

Common fingerprint types:
- **MACCS keys:** 166 predefined structural keys (e.g., "contains a 6-membered ring")
- **Morgan (ECFP):** Circular fingerprints that capture the neighborhood around each atom. ECFP4 uses radius=2.
- **RDKit fingerprint:** Topological paths of defined length
- **FCFP:** Like Morgan but uses pharmacophoric atom types

**Similarity metric -- Tanimoto coefficient:**
$$T(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{c}{a + b - c}$$
where $a$, $b$ are the number of bits set in each fingerprint, and $c$ is the number of common bits.

In [ ]:
# Compute pairwise Tanimoto similarity using Morgan fingerprints

if RDKIT_AVAILABLE:
    # Generate Morgan fingerprints (ECFP4)
    fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) for mol in mols]

    # Compute pairwise Tanimoto similarity
    n = len(fps)
    sim_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

    df_sim = pd.DataFrame(sim_matrix, index=names, columns=names)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_yticklabels(names)
    ax.set_title('Tanimoto Similarity Matrix (Morgan FP, radius=2)')
    
    # Add text annotations
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                    fontsize=9, color='black' if sim_matrix[i, j] < 0.7 else 'white')
    
    plt.colorbar(im, label='Tanimoto Similarity')
    plt.tight_layout()
    plt.show()
else:
    print("[Would display a 6x6 heatmap of Tanimoto similarities between drug molecules]")
    print("Aspirin and Paracetamol would show moderate similarity (~0.2-0.3).")
    print("Each molecule has Tanimoto=1.0 with itself.")

### 6.4 Drug-Likeness: Lipinski's Rule of Five

In 1997, Christopher Lipinski analyzed successful oral drugs and found that poor absorption is more likely when:

1. **Molecular weight > 500 Da**
2. **LogP > 5** (too lipophilic)
3. **H-bond donors > 5**
4. **H-bond acceptors > 10**

A molecule violating **two or more** rules is unlikely to be orally bioavailable.

**Beyond Lipinski -- additional filters:**
- **Veber rules:** Rotatable bonds ≤ 10, TPSA ≤ 140 Å² (oral bioavailability)
- **PAINS filters:** Pan-Assay Interference Compounds (promiscuous binders, false positives)
- **Brenk filters:** Unwanted substructures (reactive groups, toxicophores)

In [ ]:
# Check Lipinski's Rule of Five

def check_lipinski(mol):
    """Check Lipinski's Rule of Five. Returns dict with values and violation count."""
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    
    violations = sum([
        mw > 500,
        logp > 5,
        hbd > 5,
        hba > 10,
    ])
    
    return {
        'MW': round(mw, 1),
        'MW_ok': mw <= 500,
        'LogP': round(logp, 2),
        'LogP_ok': logp <= 5,
        'HBD': hbd,
        'HBD_ok': hbd <= 5,
        'HBA': hba,
        'HBA_ok': hba <= 10,
        'violations': violations,
        'drug_like': violations < 2,
    }

if RDKIT_AVAILABLE:
    print("Lipinski's Rule of Five Analysis:")
    print("=" * 75)
    for name, mol in zip(names, mols):
        result = check_lipinski(mol)
        status = "PASS" if result['drug_like'] else "FAIL"
        print(f"\n{name}: {status} ({result['violations']} violations)")
        print(f"  MW={result['MW']} {'ok' if result['MW_ok'] else 'VIOLATION'}  "
              f"LogP={result['LogP']} {'ok' if result['LogP_ok'] else 'VIOLATION'}  "
              f"HBD={result['HBD']} {'ok' if result['HBD_ok'] else 'VIOLATION'}  "
              f"HBA={result['HBA']} {'ok' if result['HBA_ok'] else 'VIOLATION'}")
else:
    print("Lipinski's Rule of Five Analysis (expected output):")
    print("All 6 drugs in our set PASS Lipinski's rules (0 violations each).")
    print("This is expected -- they are all approved oral drugs.")

---
## 7. Practical Python Examples

### 7.1 RDKit: Substructure Searching

In [ ]:
# Substructure searching: find molecules containing a specific functional group

if RDKIT_AVAILABLE:
    # Define a carboxylic acid substructure query
    query = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')  # SMARTS pattern for -COOH
    
    print("Substructure search: Carboxylic acid group (-COOH)")
    print("-" * 50)
    for name, mol in zip(names, mols):
        has_match = mol.HasSubstructMatch(query)
        matches = mol.GetSubstructMatches(query)
        print(f"  {name:15s}: {'YES' if has_match else 'no ':3s}  ({len(matches)} match(es))")
    
    # Highlight the matching substructure
    print("\nHighlighting carboxylic acid groups in matching molecules:")
    matching_mols = []
    matching_names = []
    highlights = []
    for name, mol in zip(names, mols):
        matches = mol.GetSubstructMatches(query)
        if matches:
            matching_mols.append(mol)
            matching_names.append(name)
            # Flatten all match atom indices for highlighting
            highlights.append([idx for match in matches for idx in match])
    
    img = Draw.MolsToGridImage(matching_mols, molsPerRow=3, subImgSize=(350, 250),
                               legends=matching_names, highlightAtomLists=highlights)
    display(img)
else:
    print("Substructure search: Carboxylic acid group (-COOH)")
    print("  Aspirin:        YES  (1 match)")
    print("  Ibuprofen:      YES  (1 match)")
    print("  Caffeine:       no   (0 matches)")
    print("  Paracetamol:    no   (0 matches)")
    print("  Penicillin G:   YES  (1 match)")
    print("  Metformin:      no   (0 matches)")

### 7.2 RDKit: Chemical Space Visualization

In [ ]:
# Visualize chemical space using molecular descriptors

if RDKIT_AVAILABLE:
    # Expanded drug set for a richer visualization
    drug_library = {
        'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
        'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
        'Caffeine': 'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
        'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
        'Metformin': 'CN(C)C(=N)NC(=N)N',
        'Diazepam': 'CN1C(=O)CN=C(c2ccccc21)c3ccc(Cl)cc3',
        'Omeprazole': 'COc1ccc2[nH]c(nc2c1)S(=O)Cc3ncc(C)c(OC)c3C',
        'Atorvastatin': 'CC(C)c1n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c(c2ccc(F)cc2)c(c1c3ccccc3)C(=O)Nc4ccccc4',
        'Ciprofloxacin': 'O=C(O)c1cn(C2CC2)c3cc(N4CCNCC4)c(F)cc3c1=O',
        'Doxorubicin': 'COc1cccc2c1C(=O)c3c(O)c4CC(O)(CC(OC5CC(N)C(O)C(C)O5)c4c(O)c3C2=O)C(=O)CO',
    }

    mw_list, logp_list, tpsa_list, name_list = [], [], [], []
    for name, smi in drug_library.items():
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mw_list.append(Descriptors.MolWt(mol))
            logp_list.append(Descriptors.MolLogP(mol))
            tpsa_list.append(Descriptors.TPSA(mol))
            name_list.append(name)

    fig, ax = plt.subplots(figsize=(12, 7))
    scatter = ax.scatter(mw_list, logp_list, c=tpsa_list, s=150,
                         cmap='coolwarm', edgecolors='black', linewidths=1, zorder=5)

    for i, name in enumerate(name_list):
        ax.annotate(name, (mw_list[i], logp_list[i]),
                    textcoords='offset points', xytext=(8, 5), fontsize=9)

    # Lipinski boundaries
    ax.axvline(x=500, color='red', linestyle='--', alpha=0.5, label='MW = 500')
    ax.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='LogP = 5')

    ax.set_xlabel('Molecular Weight (Da)')
    ax.set_ylabel('LogP')
    ax.set_title('Chemical Space: Drug-Likeness Landscape')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, label='TPSA (Å²)')
    plt.tight_layout()
    plt.show()

    print("The Lipinski 'drug-like' space is in the lower-left quadrant (MW<500, LogP<5).")
    print("Color encodes TPSA: blue = low (lipophilic), red = high (polar).")
else:
    print("[Would display a scatter plot of 10 drugs in MW vs LogP space, colored by TPSA]")

### 7.3 MDAnalysis: Trajectory Analysis Concepts

[MDAnalysis](https://www.mdanalysis.org/) is a Python library for analyzing MD trajectories. Here we demonstrate the key concepts using simulated data.

In [ ]:
# MDAnalysis concept demonstration using synthetic data
# In practice, you would load actual trajectory files:
#   import MDAnalysis as mda
#   u = mda.Universe('topology.gro', 'trajectory.xtc')
#   protein = u.select_atoms('protein')
#   for ts in u.trajectory:
#       rmsd = ...

# Simulate a distance monitoring analysis (e.g., salt bridge distance)
time = np.linspace(0, 50, 2500)  # 50 ns

# Two salt bridges: one stable, one transient
# Stable salt bridge (Asp-Arg, ~3 Å)
dist_stable = 3.0 + 0.3 * np.random.randn(len(time))
dist_stable = np.clip(dist_stable, 2.0, None)

# Transient salt bridge (forms and breaks)
# Use a simple two-state model
dist_transient = np.zeros(len(time))
state = 0  # 0 = broken, 1 = formed
for i in range(len(time)):
    if state == 0 and np.random.random() < 0.005:
        state = 1
    elif state == 1 and np.random.random() < 0.02:
        state = 0
    dist_transient[i] = 3.2 + 0.4*np.random.randn() if state == 1 else 8.0 + 1.5*np.random.randn()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(time, dist_stable, 'b-', linewidth=0.5, alpha=0.7)
axes[0].axhline(y=4.0, color='red', linestyle='--', alpha=0.7, label='Salt bridge cutoff (4 Å)')
axes[0].set_ylabel('Distance (Å)')
axes[0].set_title('Stable Salt Bridge: Asp52--Arg87')
axes[0].set_ylim(1, 12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(time, dist_transient, 'g-', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=4.0, color='red', linestyle='--', alpha=0.7, label='Salt bridge cutoff (4 Å)')
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Distance (Å)')
axes[1].set_title('Transient Salt Bridge: Glu23--Lys105')
axes[1].set_ylim(1, 15)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

pct_formed_stable = (dist_stable < 4.0).mean() * 100
pct_formed_transient = (dist_transient < 4.0).mean() * 100
print(f"Stable salt bridge: formed {pct_formed_stable:.1f}% of the time")
print(f"Transient salt bridge: formed {pct_formed_transient:.1f}% of the time")

In [ ]:
# PDB file parsing: extract basic structural information
# This demonstrates how to work with PDB files in pure Python

sample_pdb = """HEADER    HYDROLASE                               01-JAN-00   1ABC
ATOM      1  N   ALA A   1      27.340  24.430   2.614  1.00  9.67           N
ATOM      2  CA  ALA A   1      26.266  25.413   2.842  1.00 10.38           C
ATOM      3  C   ALA A   1      26.913  26.639   3.531  1.00  9.62           C
ATOM      4  O   ALA A   1      27.886  26.463   4.263  1.00  9.62           O
ATOM      5  CB  ALA A   1      25.112  24.880   3.649  1.00 13.77           C
ATOM      6  N   GLY A   2      26.335  27.770   3.258  1.00  9.27           N
ATOM      7  CA  GLY A   2      26.850  29.021   3.898  1.00  9.87           C
ATOM      8  C   GLY A   2      26.100  29.300   5.202  1.00 10.23           C
ATOM      9  O   GLY A   2      24.865  29.024   5.330  1.00 10.65           O
ATOM     10  N   VAL A   3      26.849  29.853   6.162  1.00 10.12           N
ATOM     11  CA  VAL A   3      26.235  30.340   7.410  1.00 10.53           C
ATOM     12  C   VAL A   3      26.882  31.680   7.745  1.00  9.93           C
ATOM     13  O   VAL A   3      28.094  31.842   7.653  1.00 10.25           O
ATOM     14  CB  VAL A   3      26.344  29.420   8.665  1.00 12.10           C
ATOM     15  CG1 VAL A   3      25.621  29.991   9.852  1.00 13.45           C
ATOM     16  CG2 VAL A   3      25.817  28.015   8.352  1.00 12.88           C
END
"""

def parse_pdb_atoms(pdb_text):
    """Parse ATOM records from PDB text into a list of dicts."""
    atoms = []
    for line in pdb_text.strip().split('\n'):
        if line.startswith('ATOM'):
            atoms.append({
                'serial': int(line[6:11]),
                'name': line[12:16].strip(),
                'resname': line[17:20].strip(),
                'chain': line[21],
                'resid': int(line[22:26]),
                'x': float(line[30:38]),
                'y': float(line[38:46]),
                'z': float(line[46:54]),
                'bfactor': float(line[60:66]),
                'element': line[76:78].strip(),
            })
    return atoms

atoms = parse_pdb_atoms(sample_pdb)
df_atoms = pd.DataFrame(atoms)
print("Parsed PDB atoms:")
print(df_atoms.to_string(index=False))

# Calculate distances between CA atoms
ca_atoms = df_atoms[df_atoms['name'] == 'CA']
print("\nCA atom coordinates:")
for _, row in ca_atoms.iterrows():
    print(f"  {row['resname']}{row['resid']}: ({row['x']:.1f}, {row['y']:.1f}, {row['z']:.1f})")

# Distance between first and last CA
ca_coords = ca_atoms[['x', 'y', 'z']].values
dist = np.linalg.norm(ca_coords[-1] - ca_coords[0])
print(f"\nCA-CA distance (residue 1 to 3): {dist:.2f} Å")

### 7.4 Simulating a Simple 2D Molecular Dynamics System

In [ ]:
# Simple 2D Lennard-Jones MD simulation
# Demonstrates the core concepts of MD: forces, integration, PBC, thermostatting

def lj_force_2d(positions, box_size, epsilon=1.0, sigma=1.0, r_cutoff=2.5):
    """Compute pairwise LJ forces with PBC and minimum image convention."""
    n = len(positions)
    forces = np.zeros_like(positions)
    potential = 0.0
    
    for i in range(n):
        for j in range(i + 1, n):
            dr = positions[j] - positions[i]
            # Minimum image convention
            dr -= box_size * np.round(dr / box_size)
            r = np.linalg.norm(dr)
            
            if r < r_cutoff and r > 0.5:
                sr6 = (sigma / r)**6
                sr12 = sr6**2
                f_mag = 24 * epsilon * (2 * sr12 - sr6) / r
                f_vec = f_mag * dr / r
                forces[i] -= f_vec
                forces[j] += f_vec
                potential += 4 * epsilon * (sr12 - sr6)
    
    return forces, potential

def run_md_2d(n_particles=16, box_size=8.0, temperature=1.0, dt=0.005, n_steps=2000):
    """Run a simple 2D Lennard-Jones MD simulation."""
    # Initialize on a grid
    n_side = int(np.ceil(np.sqrt(n_particles)))
    spacing = box_size / n_side
    positions = []
    for i in range(n_side):
        for j in range(n_side):
            if len(positions) < n_particles:
                positions.append([spacing * (i + 0.5), spacing * (j + 0.5)])
    positions = np.array(positions)
    
    # Random initial velocities (Maxwell-Boltzmann)
    velocities = np.random.randn(n_particles, 2) * np.sqrt(temperature)
    velocities -= velocities.mean(axis=0)  # Remove center-of-mass motion
    
    # Storage
    trajectory = [positions.copy()]
    energies = []
    temperatures = []
    
    forces, pe = lj_force_2d(positions, box_size)
    
    for step in range(n_steps):
        # Velocity Verlet
        velocities += 0.5 * forces * dt
        positions += velocities * dt
        # Apply PBC
        positions = positions % box_size
        forces, pe = lj_force_2d(positions, box_size)
        velocities += 0.5 * forces * dt
        
        # Simple velocity rescaling thermostat (every 50 steps)
        if step % 50 == 0:
            ke = 0.5 * np.sum(velocities**2)
            current_T = ke / n_particles
            if current_T > 0:
                scale = np.sqrt(temperature / current_T)
                velocities *= scale
        
        ke = 0.5 * np.sum(velocities**2)
        energies.append(pe + ke)
        temperatures.append(ke / n_particles)
        
        if step % 20 == 0:
            trajectory.append(positions.copy())
    
    return trajectory, energies, temperatures

traj, energies, temps = run_md_2d(n_particles=25, box_size=10.0, temperature=0.8, n_steps=3000)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Snapshot: initial and final
for pos in [traj[0]]:
    axes[0].scatter(pos[:, 0], pos[:, 1], s=200, c='lightblue', edgecolors='blue',
                    linewidths=1.5, alpha=0.5, label='Initial')
for pos in [traj[-1]]:
    axes[0].scatter(pos[:, 0], pos[:, 1], s=200, c='salmon', edgecolors='red',
                    linewidths=1.5, alpha=0.7, label='Final')
axes[0].set_xlim(0, 10)
axes[0].set_ylim(0, 10)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('2D LJ System: Initial vs Final')
axes[0].set_aspect('equal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Energy
axes[1].plot(energies, 'b-', linewidth=0.5)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Total Energy')
axes[1].set_title('Total Energy vs. Time')
axes[1].grid(True, alpha=0.3)

# Temperature
axes[2].plot(temps, 'r-', linewidth=0.5, alpha=0.7)
axes[2].axhline(y=0.8, color='black', linestyle='--', label='Target T = 0.8')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Temperature (reduced units)')
axes[2].set_title('Temperature vs. Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("This simple 2D MD illustrates the core workflow:")
print("  1. Initialize positions and velocities")
print("  2. Compute forces from the potential (Lennard-Jones)")
print("  3. Integrate equations of motion (Velocity Verlet)")
print("  4. Apply boundary conditions (PBC)")
print("  5. Control temperature (velocity rescaling thermostat)")

---
## 8. Summary and Key Takeaways

| Topic | Key Concepts | Main Tools |
|-------|-------------|------------|
| **Molecular Modeling** | Levels of theory (QM → MM → CG) | GROMACS, NAMD, AMBER |
| **Force Fields** | Bonded + non-bonded terms; parameterization | AMBER, CHARMM, OPLS |
| **Homology Modeling** | Template → alignment → model → validation | SWISS-MODEL, Modeller, AlphaFold |
| **Molecular Docking** | Pose prediction, scoring, virtual screening | AutoDock Vina, Glide, GOLD |
| **Molecular Dynamics** | Integration, PBC, thermostats/barostats | GROMACS, OpenMM |
| **MD Analysis** | RMSD, RMSF, Rg, H-bonds, PCA | MDAnalysis, pytraj, GROMACS |
| **Chemoinformatics** | SMILES, fingerprints, Lipinski, similarity | RDKit |

**Best practices:**
1. Always validate your models (Ramachandran, QMEAN for homology; convergence checks for MD)
2. Never mix force field parameters from different families
3. Docking scores are approximate -- always validate top hits experimentally
4. Check MD equilibration before analyzing production data
5. Use appropriate statistics: run replicates, compute error bars

---
## 9. Exercises

### Exercise 1: Force Field Energy Calculation

Given the following bond parameters, calculate the total bond stretching energy for a simple molecule:

- Bond C1-C2: r = 1.54 Å, r₀ = 1.52 Å, k_b = 317 kcal/mol/Å²
- Bond C2-C3: r = 1.50 Å, r₀ = 1.52 Å, k_b = 317 kcal/mol/Å²  
- Bond C2-O4: r = 1.43 Å, r₀ = 1.41 Å, k_b = 320 kcal/mol/Å²

Also compute the Lennard-Jones interaction energy between two carbon atoms at distance 4.0 Å with σ = 3.4 Å and ε = 0.086 kcal/mol.

In [ ]:
# Your solution here


In [ ]:
# --- Solution ---

def bond_energy(r, r0, kb):
    return 0.5 * kb * (r - r0)**2

bonds = [
    {'name': 'C1-C2', 'r': 1.54, 'r0': 1.52, 'kb': 317},
    {'name': 'C2-C3', 'r': 1.50, 'r0': 1.52, 'kb': 317},
    {'name': 'C2-O4', 'r': 1.43, 'r0': 1.41, 'kb': 320},
]

total_bond_E = 0
for bond in bonds:
    E = bond_energy(bond['r'], bond['r0'], bond['kb'])
    total_bond_E += E
    print(f"{bond['name']}: E = 0.5 * {bond['kb']} * ({bond['r']} - {bond['r0']})² = {E:.4f} kcal/mol")

print(f"\nTotal bond stretching energy: {total_bond_E:.4f} kcal/mol")

# Lennard-Jones
r, sigma, epsilon = 4.0, 3.4, 0.086
E_lj = 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)
print(f"\nLJ energy at r={r} Å: {E_lj:.6f} kcal/mol")
print(f"(Negative = attractive interaction at this distance)")

### Exercise 2: SMILES and Molecular Properties

Using RDKit (or by hand if RDKit is not available):

1. Write the SMILES for the following molecules: ethanol, benzene, acetic acid, glycine
2. Create RDKit molecule objects from your SMILES
3. Calculate MW, LogP, number of H-bond donors/acceptors for each
4. Check which molecules satisfy Lipinski's Rule of Five
5. Which molecule is most lipophilic? Which is most polar?

In [ ]:
# Your solution here


In [ ]:
# --- Solution ---

exercise_molecules = {
    'Ethanol': 'CCO',
    'Benzene': 'c1ccccc1',
    'Acetic acid': 'CC(=O)O',
    'Glycine': 'NCC(=O)O',
}

if RDKIT_AVAILABLE:
    print(f"{'Molecule':<15} {'SMILES':<15} {'MW':>6} {'LogP':>6} {'HBD':>4} {'HBA':>4} {'TPSA':>6} {'Lipinski':>8}")
    print("-" * 72)
    
    for name, smi in exercise_molecules.items():
        mol = Chem.MolFromSmiles(smi)
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        tpsa = Descriptors.TPSA(mol)
        lip = check_lipinski(mol)
        print(f"{name:<15} {smi:<15} {mw:6.1f} {logp:6.2f} {hbd:4d} {hba:4d} {tpsa:6.1f} {'PASS' if lip['drug_like'] else 'FAIL':>8}")
    
    print("\nAll four molecules easily pass Lipinski's rules (they are very small).")
    print("Benzene is most lipophilic (highest LogP).")
    print("Glycine is most polar (highest TPSA, lowest LogP).")
else:
    print("Expected results:")
    print("Ethanol     CCO          46.1  -0.31   1    1   20.2     PASS")
    print("Benzene     c1ccccc1     78.1   1.56   0    0    0.0     PASS")
    print("Acetic acid CC(=O)O      60.1  -0.17   1    2   37.3     PASS")
    print("Glycine     NCC(=O)O     75.1  -1.03   2    3   63.3     PASS")

### Exercise 3: Verlet Integrator for a Morse Potential

The **Morse potential** is a more realistic model for a chemical bond than the harmonic potential:

$$V(r) = D_e (1 - e^{-a(r - r_e)})^2$$

where $D_e$ is the well depth, $r_e$ is the equilibrium distance, and $a$ controls the width.

1. Implement the Morse potential and its derivative (force)
2. Use the velocity Verlet integrator to simulate a diatomic molecule oscillating in a Morse potential
3. Use parameters: $D_e = 5.0$, $a = 1.0$, $r_e = 1.5$, mass = 1.0
4. Start at $r = 2.0$ with zero velocity, run for 2000 steps with $\Delta t = 0.01$
5. Plot the position over time and check energy conservation

In [ ]:
# Your solution here


In [ ]:
# --- Solution ---

def morse_potential(r, De=5.0, a=1.0, re=1.5):
    return De * (1 - np.exp(-a * (r - re)))**2

def morse_force(r, De=5.0, a=1.0, re=1.5):
    """F = -dV/dr for Morse potential."""
    return -2 * De * a * (1 - np.exp(-a * (r - re))) * np.exp(-a * (r - re))

# Velocity Verlet integration
De, a_param, re, mass = 5.0, 1.0, 1.5, 1.0
dt = 0.01
n_steps = 2000

r = np.zeros(n_steps)
v = np.zeros(n_steps)
r[0] = 2.0  # Starting position
v[0] = 0.0  # Starting velocity

acc = morse_force(r[0], De, a_param, re) / mass

for i in range(1, n_steps):
    r[i] = r[i-1] + v[i-1]*dt + 0.5*acc*dt**2
    acc_new = morse_force(r[i], De, a_param, re) / mass
    v[i] = v[i-1] + 0.5*(acc + acc_new)*dt
    acc = acc_new

t = np.arange(n_steps) * dt
KE = 0.5 * mass * v**2
PE = morse_potential(r, De, a_param, re)
TE = KE + PE

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(t, r, 'b-', linewidth=1.5)
axes[0].axhline(y=re, color='red', linestyle='--', alpha=0.5, label=f'r_e = {re}')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Bond length r')
axes[0].set_title('Anharmonic Oscillation (Morse Potential)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Phase space
axes[1].plot(r, v, 'g-', linewidth=0.5)
axes[1].set_xlabel('Position r')
axes[1].set_ylabel('Velocity')
axes[1].set_title('Phase Space (Morse -- non-circular!)')
axes[1].grid(True, alpha=0.3)

# Energy
axes[2].plot(t, KE, label='Kinetic', alpha=0.7)
axes[2].plot(t, PE, label='Potential', alpha=0.7)
axes[2].plot(t, TE, 'k-', linewidth=2, label='Total')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Energy')
axes[2].set_title('Energy Conservation')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Energy drift: {abs(TE[-1] - TE[0]):.2e}")
print(f"Unlike harmonic, Morse oscillation is asymmetric: bond stretches more than it compresses.")
print(f"Phase space trajectory is egg-shaped rather than circular.")

### Exercise 4: Virtual Screening Analysis

You have docking results for 20 compounds against a drug target. The data includes binding affinity (kcal/mol) and molecular properties. Your tasks:

1. Filter compounds that satisfy Lipinski's Rule of Five
2. Among drug-like compounds, rank by docking score
3. Apply a binding affinity cutoff of -7.0 kcal/mol
4. Identify the top 3 hits and explain why they might be good candidates
5. Plot the results

In [ ]:
# Virtual screening dataset
np.random.seed(42)
n_compounds = 20

screening_data = pd.DataFrame({
    'compound_id': [f'CPD-{i+1:03d}' for i in range(n_compounds)],
    'MW': np.random.uniform(150, 600, n_compounds).round(1),
    'LogP': np.random.uniform(-1, 7, n_compounds).round(2),
    'HBD': np.random.randint(0, 8, n_compounds),
    'HBA': np.random.randint(0, 14, n_compounds),
    'docking_score': np.random.uniform(-10, -3, n_compounds).round(2),
    'TPSA': np.random.uniform(20, 180, n_compounds).round(1),
})

print("Virtual Screening Dataset:")
print(screening_data.to_string(index=False))

In [ ]:
# Your solution here


In [ ]:
# --- Solution ---

# Step 1: Apply Lipinski filter
lipinski_mask = (
    (screening_data['MW'] <= 500).astype(int) +
    (screening_data['LogP'] <= 5).astype(int) +
    (screening_data['HBD'] <= 5).astype(int) +
    (screening_data['HBA'] <= 10).astype(int)
)
screening_data['lipinski_violations'] = 4 - lipinski_mask
screening_data['drug_like'] = screening_data['lipinski_violations'] < 2

drug_like = screening_data[screening_data['drug_like']].copy()
print(f"Drug-like compounds (Lipinski): {len(drug_like)}/{n_compounds}")

# Step 2: Rank by docking score
drug_like_sorted = drug_like.sort_values('docking_score')

# Step 3: Apply affinity cutoff
hits = drug_like_sorted[drug_like_sorted['docking_score'] <= -7.0]
print(f"Hits (score <= -7.0 kcal/mol): {len(hits)}")

# Step 4: Top 3
top3 = drug_like_sorted.head(3)
print("\nTop 3 Hits:")
print(top3[['compound_id', 'docking_score', 'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'lipinski_violations']].to_string(index=False))

# Step 5: Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# All compounds colored by drug-likeness
colors = ['green' if dl else 'red' for dl in screening_data['drug_like']]
axes[0].scatter(screening_data['MW'], screening_data['docking_score'],
                c=colors, s=100, edgecolors='black', alpha=0.7)
axes[0].axhline(y=-7.0, color='blue', linestyle='--', label='Score cutoff (-7.0)')
axes[0].axvline(x=500, color='red', linestyle='--', alpha=0.5, label='MW = 500')
axes[0].set_xlabel('Molecular Weight (Da)')
axes[0].set_ylabel('Docking Score (kcal/mol)')
axes[0].set_title('Virtual Screening Results\n(green = drug-like, red = not drug-like)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Annotate top 3
for _, row in top3.iterrows():
    axes[0].annotate(row['compound_id'], (row['MW'], row['docking_score']),
                     textcoords='offset points', xytext=(5, 5), fontsize=9, fontweight='bold')

# Score distribution
axes[1].hist(screening_data['docking_score'], bins=10, color='lightblue', edgecolor='navy', alpha=0.7, label='All')
axes[1].hist(hits['docking_score'], bins=5, color='green', edgecolor='darkgreen', alpha=0.7, label='Hits')
axes[1].axvline(x=-7.0, color='red', linestyle='--', label='Cutoff')
axes[1].set_xlabel('Docking Score (kcal/mol)')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 5: Analyzing MD Equilibration

Proper equilibration is critical before collecting MD production data. In this exercise, you will analyze synthetic MD data to determine:

1. When does the RMSD equilibrate? (Define equilibration as when the running average changes by less than 0.005 nm over 5 ns)
2. Calculate the average RMSD and standard deviation during the equilibrated phase
3. Is the protein stable? (Check both RMSD and Rg stability)
4. Identify the most flexible residues from RMSF data

In [ ]:
# Your solution here


In [ ]:
# --- Solution ---

# Generate synthetic MD data
np.random.seed(123)
time_ns = np.linspace(0, 100, 10000)
dt_ns = time_ns[1] - time_ns[0]

# RMSD with equilibration phase (~15 ns)
rmsd = 0.22 * (1 - np.exp(-time_ns / 7)) + 0.015 * np.random.randn(len(time_ns))
rmsd = np.clip(rmsd, 0.01, None)

# Rg (stable)
rg = 1.52 + 0.01 * np.random.randn(len(time_ns))

# 1. Determine equilibration time
window = 500  # ~5 ns
rmsd_running = np.convolve(rmsd, np.ones(window)/window, mode='valid')
time_running = time_ns[window-1:]

# Find where the derivative of the running average becomes small
deriv = np.abs(np.diff(rmsd_running) / dt_ns)
threshold = 0.005  # nm per ns

# Check where the derivative stays below threshold for an extended period
equil_window = 500
equil_idx = None
for i in range(len(deriv) - equil_window):
    if np.all(deriv[i:i+equil_window] < threshold):
        equil_idx = i
        break

equil_time = time_running[equil_idx] if equil_idx is not None else 0
print(f"1. RMSD equilibrates at approximately {equil_time:.1f} ns")

# 2. Statistics during equilibrated phase
equil_mask = time_ns >= equil_time
rmsd_equil = rmsd[equil_mask]
print(f"\n2. Equilibrated RMSD:")
print(f"   Mean = {rmsd_equil.mean():.3f} nm")
print(f"   Std  = {rmsd_equil.std():.3f} nm")

# 3. Stability assessment
rg_equil = rg[equil_mask]
rg_drift = abs(rg_equil[-500:].mean() - rg_equil[:500].mean())
rmsd_drift = abs(rmsd_equil[-500:].mean() - rmsd_equil[:500].mean())
print(f"\n3. Stability assessment:")
print(f"   RMSD drift (first vs last 5 ns of production): {rmsd_drift:.4f} nm")
print(f"   Rg drift: {rg_drift:.4f} nm")
print(f"   Protein is {'STABLE' if rmsd_drift < 0.05 and rg_drift < 0.02 else 'UNSTABLE'}")

# 4. RMSF analysis
n_res = 120
res_ids = np.arange(1, n_res + 1)
rmsf = 0.06 + 0.02 * np.random.rand(n_res)
# Add flexible regions
rmsf[:4] = 0.3 + 0.05 * np.random.rand(4)  # N-term
rmsf[-3:] = 0.35 + 0.05 * np.random.rand(3)  # C-term
rmsf[35:45] = 0.2 + 0.04 * np.random.rand(10)  # Loop
rmsf[72:80] = 0.18 + 0.03 * np.random.rand(8)  # Loop

flexible_threshold = 0.15
flexible_residues = res_ids[rmsf > flexible_threshold]
print(f"\n4. Flexible residues (RMSF > {flexible_threshold} nm): {flexible_residues.tolist()}")
print(f"   These correspond to: N-terminus, loops (36-44, 73-79), and C-terminus")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(time_ns, rmsd, 'b-', linewidth=0.3, alpha=0.5)
axes[0].plot(time_running, rmsd_running, 'r-', linewidth=2)
axes[0].axvline(x=equil_time, color='green', linewidth=2, linestyle='--',
                label=f'Equilibration: {equil_time:.0f} ns')
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('RMSD (nm)')
axes[0].set_title('RMSD Equilibration Analysis')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_ns, rg, 'g-', linewidth=0.3, alpha=0.5)
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Rg (nm)')
axes[1].set_title('Radius of Gyration')
axes[1].grid(True, alpha=0.3)

axes[2].bar(res_ids, rmsf, color='purple', alpha=0.7, width=1.0)
axes[2].axhline(y=flexible_threshold, color='red', linestyle='--', label=f'Threshold ({flexible_threshold} nm)')
axes[2].set_xlabel('Residue Number')
axes[2].set_ylabel('RMSF (nm)')
axes[2].set_title('Per-Residue Flexibility')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 6: Tanimoto Similarity and Clustering

Given a set of SMILES strings, compute pairwise Tanimoto similarities using Morgan fingerprints and identify clusters of similar compounds. If RDKit is not installed, implement a simplified Tanimoto coefficient on binary vectors.

Task:
1. Compute the Tanimoto similarity between all pairs of molecules
2. Find the most similar pair and the most dissimilar pair
3. Using a similarity threshold of 0.3, group compounds into clusters

In [ ]:
# Your solution here


In [ ]:
# --- Solution ---

# If RDKit is not available, demonstrate with simple binary fingerprints

def tanimoto_binary(fp1, fp2):
    """Compute Tanimoto coefficient between two binary arrays."""
    both = np.sum(fp1 & fp2)
    either = np.sum(fp1 | fp2)
    return both / either if either > 0 else 0.0

if RDKIT_AVAILABLE:
    cluster_smiles = {
        'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
        'Salicylic acid': 'OC(=O)c1ccccc1O',  # Similar to aspirin
        'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
        'Naproxen': 'COc1ccc2cc(ccc2c1)C(C)C(=O)O',  # Similar to ibuprofen
        'Caffeine': 'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
        'Theophylline': 'Cn1c2c(c(=O)n(c1=O)C)[nH]cn2',  # Similar to caffeine
        'Metformin': 'CN(C)C(=N)NC(=N)N',
    }
    
    c_mols = {}
    c_fps = {}
    for name, smi in cluster_smiles.items():
        mol = Chem.MolFromSmiles(smi)
        if mol:
            c_mols[name] = mol
            c_fps[name] = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    
    c_names = list(c_fps.keys())
    n = len(c_names)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = DataStructs.TanimotoSimilarity(c_fps[c_names[i]], c_fps[c_names[j]])
else:
    # Simplified demo with random binary fingerprints
    c_names = ['Aspirin', 'Salicylic acid', 'Ibuprofen', 'Naproxen', 'Caffeine', 'Theophylline', 'Metformin']
    np.random.seed(42)
    n = len(c_names)
    fps = np.random.randint(0, 2, (n, 100))
    # Make similar pairs share more bits
    fps[1] = fps[0].copy(); fps[1, np.random.choice(100, 30, replace=False)] ^= 1
    fps[3] = fps[2].copy(); fps[3, np.random.choice(100, 35, replace=False)] ^= 1
    fps[5] = fps[4].copy(); fps[5, np.random.choice(100, 25, replace=False)] ^= 1
    
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = tanimoto_binary(fps[i], fps[j])

# Find most similar and dissimilar pairs
max_sim, max_pair = 0, (None, None)
min_sim, min_pair = 1, (None, None)
for i in range(n):
    for j in range(i+1, n):
        if sim[i, j] > max_sim:
            max_sim = sim[i, j]
            max_pair = (c_names[i], c_names[j])
        if sim[i, j] < min_sim:
            min_sim = sim[i, j]
            min_pair = (c_names[i], c_names[j])

print(f"Most similar pair:    {max_pair[0]} -- {max_pair[1]} (Tanimoto = {max_sim:.3f})")
print(f"Most dissimilar pair: {min_pair[0]} -- {min_pair[1]} (Tanimoto = {min_sim:.3f})")

# Simple single-linkage clustering with threshold
threshold = 0.3
clusters = list(range(n))
for i in range(n):
    for j in range(i+1, n):
        if sim[i, j] >= threshold:
            old_cluster = clusters[j]
            new_cluster = clusters[i]
            for k in range(n):
                if clusters[k] == old_cluster:
                    clusters[k] = new_cluster

print(f"\nClusters (threshold = {threshold}):")
for cluster_id in sorted(set(clusters)):
    members = [c_names[i] for i in range(n) if clusters[i] == cluster_id]
    print(f"  Cluster {cluster_id}: {', '.join(members)}")

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(c_names, rotation=45, ha='right')
ax.set_yticklabels(c_names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='black' if sim[i,j] < 0.6 else 'white')
ax.set_title('Pairwise Tanimoto Similarity')
plt.colorbar(im)
plt.tight_layout()
plt.show()

---
## 10. Further Reading and Resources

### Textbooks
- Leach, A.R. *Molecular Modelling: Principles and Applications* (2001) -- Comprehensive introduction
- Frenkel, D. & Smit, B. *Understanding Molecular Simulation* (2002) -- Statistical mechanics foundations
- Schlick, T. *Molecular Modeling and Simulation* (2010) -- Mathematical rigor

### Online Resources
- [GROMACS tutorials by Justin Lemkul](http://www.mdtutorials.com/gmx/) -- Step-by-step MD tutorials
- [RDKit Cookbook](https://www.rdkit.org/docs/Cookbook.html) -- Chemoinformatics recipes
- [MDAnalysis User Guide](https://userguide.mdanalysis.org/) -- Trajectory analysis
- [SWISS-MODEL](https://swissmodel.expasy.org/) -- Automated homology modeling
- [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) -- Predicted structures

### Software
- **GROMACS:** Free, open-source MD engine (http://www.gromacs.org)
- **AutoDock Vina:** Free docking software (https://vina.scripps.edu)
- **PyMOL:** Molecular visualization (https://pymol.org)
- **RDKit:** Open-source chemoinformatics (https://www.rdkit.org)
- **MDAnalysis:** Python trajectory analysis (https://www.mdanalysis.org)
- **OpenMM:** GPU-accelerated MD with Python API (https://openmm.org)

---[< Previous: From Sequence to Discovery: An Integrative Bioi...](../08_Capstone_Project/01_capstone_project.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Molecular Modeling + Docking: Overview >](01_molecular_modeling_and_docking.ipynb)---